## **Comparing Machine Learning and Statistical Models for FIFA World Cup Prediction**

Everything is fit on 358 real international matches: every game from the 2010–2022 World Cups (256 matches) plus the 2020 and 2024 European Championships (102), pulled from the openfootball project, specifically its worldcup.json and euro.json datasets, which are dedicated to the public domain. The classifiers learn a mapping from match features to results on these games; the rating systems are computed directly from the results graph. The field is the real, confirmed 2026 draw, 48 teams, 12 groups.

In [23]:
#packages
!pip -q install pandas numpy matplotlib seaborn scikit-learn xgboost requests

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
import os
import re

from collections import Counter

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries successfully imported.")

Libraries successfully imported.


In [25]:
#project configuration

RANDOM_STATE = 42

DATA_DIR = "/content/data"
RAW_DIR = f"{DATA_DIR}/raw"
PROCESSED_DIR = f"{DATA_DIR}/processed"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Project directories created.")

Project directories created.


In [26]:
#URLs for the openfootball datasets

# OpenFootball JSON repositories
BASE_WC = "https://raw.githubusercontent.com/openfootball/worldcup.json/master"
BASE_EURO = "https://raw.githubusercontent.com/openfootball/euro.json/master"

# Historical tournaments we want
WORLD_CUPS = [2010, 2014, 2018, 2022]
EUROS = [2020, 2024]

print("World Cups:", WORLD_CUPS)
print("European Championships:", EUROS)

World Cups: [2010, 2014, 2018, 2022]
European Championships: [2020, 2024]


# **WorldCup Data**

---



In [27]:
url = f"{BASE_WC}/2022/worldcup.json"

response = requests.get(url)

print("Status code:", response.status_code)
print("File size:", len(response.text), "characters")

Status code: 200
File size: 25297 characters


In [28]:
wc2022 = response.json()

print(type(wc2022))
print(wc2022.keys())

<class 'dict'>
dict_keys(['name', 'matches'])


In [29]:
wc2022["matches"][0]

{'round': 'Matchday 1',
 'date': '2022-11-20',
 'time': '19:00',
 'team1': 'Qatar',
 'team2': 'Ecuador',
 'score': {'ft': [0, 2], 'ht': [0, 2]},
 'goals1': [],
 'goals2': [{'name': 'Enner Valencia', 'minute': '16', 'penalty': True},
  {'name': 'Enner Valencia', 'minute': '31'}],
 'group': 'Group A',
 'ground': 'Al Bayt Stadium, Al Khor'}

In [30]:
def load_openfootball_tournament(url, tournament_name, year):
    """
    Download and convert an OpenFootball JSON tournament
    into a match-level DataFrame.
    """

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    matches = []

    for match in data["matches"]:

        # Some records may not have a completed score
        if "score" not in match:
            continue

        score = match["score"]

        # We want the full-time result
        if "ft" not in score:
            continue

        ft_score = score["ft"]

        if len(ft_score) != 2:
            continue

        matches.append({
            "date": match.get("date"),
            "tournament": tournament_name,
            "year": year,
            "round": match.get("round"),
            "team1": match.get("team1"),
            "team2": match.get("team2"),
            "goals1": ft_score[0],
            "goals2": ft_score[1],
            "group": match.get("group"),
            "ground": match.get("ground")
        })

    return pd.DataFrame(matches)

In [31]:
test_2022 = load_openfootball_tournament(
    f"{BASE_WC}/2022/worldcup.json",
    "World Cup",
    2022
)

print("Matches:", len(test_2022))

display(test_2022.head())

Matches: 57


,date,tournament,year,round,team1,team2,goals1,goals2,group,ground
0,2022-11-20,World Cup,2022,Matchday 1,Qatar,Ecuador,0,2,Group A,"Al Bayt Stadium, Al Khor"
1,2022-11-21,World Cup,2022,Matchday 2,Senegal,Netherlands,0,2,Group A,"Al Thumama Stadium, Doha"
2,2022-11-25,World Cup,2022,Matchday 6,Qatar,Senegal,1,3,Group A,"Al Thumama Stadium, Doha"
3,2022-11-25,World Cup,2022,Matchday 6,Netherlands,Ecuador,1,1,Group A,"Khalifa International Stadium, Al Rayyan"
4,2022-11-29,World Cup,2022,Matchday 10,Ecuador,Senegal,1,2,Group A,"Khalifa International Stadium, Al Rayyan"


In [32]:
world_cup_dfs = []

for year in WORLD_CUPS:

    url = f"{BASE_WC}/{year}/worldcup.json"

    df = load_openfootball_tournament(
        url,
        "World Cup",
        year
    )

    print(f"{year}: {len(df)} matches")

    world_cup_dfs.append(df)

world_cups = pd.concat(
    world_cup_dfs,
    ignore_index=True
)

print("\nTotal World Cup matches:", len(world_cups))

2010: 4 matches
2014: 20 matches
2018: 64 matches
2022: 57 matches

Total World Cup matches: 145


# **EuroCup Data**

In [33]:
euro_dfs = []

for year in EUROS:

    url = f"{BASE_EURO}/{year}/euro.json"

    df = load_openfootball_tournament(
        url,
        "European Championship",
        year
    )

    print(f"Euro {year}: {len(df)} matches")

    euro_dfs.append(df)

euros = pd.concat(
    euro_dfs,
    ignore_index=True
)

print("\nTotal Euro matches:", len(euros))

Euro 2020: 51 matches
Euro 2024: 51 matches

Total Euro matches: 102


# **Combined Data**

In [ ]:
matches_men = pd.concat(
    [world_cups, euros],
    ignore_index=True
)

print("Total matches:", len(matches_men))

In [ ]:
matches_men = matches_men.rename(columns={
    "team1": "team_a",
    "team2": "team_b",
    "goals1": "score_a",
    "goals2": "score_b"
})

matches_men["date"] = pd.to_datetime(
    matches_men["date"]
)

matches_men["total_goals"] = (
    matches_men["score_a"] +
    matches_men["score_b"]
)

matches_men["goal_difference"] = (
    matches_men["score_a"] -
    matches_men["score_b"]
)

matches_men.head()

In [ ]:
def get_result(row):

    if row["score_a"] > row["score_b"]:
        return "Team A Win"

    elif row["score_a"] < row["score_b"]:
        return "Team B Win"

    else:
        return "Draw"


matches_men["result"] = matches_men.apply(
    get_result,
    axis=1
)

matches_men["result"].value_counts()

In [ ]:
def load_openfootball_tournament(url, tournament_name, year):

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    matches = []

    for match in data["matches"]:

        if "score" not in match:
            continue

        score = match["score"]

        if "ft" not in score:
            continue

        ft_score = score["ft"]

        if len(ft_score) != 2:
            continue

        penalty_score = score.get("p")

        matches.append({
            "date": match.get("date"),
            "tournament": tournament_name,
            "year": year,
            "round": match.get("round"),
            "team1": match.get("team1"),
            "team2": match.get("team2"),
            "goals1": ft_score[0],
            "goals2": ft_score[1],
            "penalty1": penalty_score[0] if penalty_score else np.nan,
            "penalty2": penalty_score[1] if penalty_score else np.nan,
            "group": match.get("group"),
            "ground": match.get("ground")
        })

    return pd.DataFrame(matches)

In [ ]:
shootouts = matches_men[
    matches_men["penalty1"].notna()
]

print("Matches decided by penalty shootout:", len(shootouts))

display(
    shootouts[
        [
            "date",
            "tournament",
            "year",
            "team_a",
            "team_b",
            "score_a",
            "score_b",
            "penalty1",
            "penalty2"
        ]
    ]
)

In [ ]:
print("=" * 50)
print("DATASET VALIDATION")
print("=" * 50)

print("Total matches:", len(matches_men))

print("\nBy tournament:")
print(
    matches_men
    .groupby(["tournament", "year"])
    .size()
)

print("\nExpected:")
print("World Cups: 256")
print("European Championships: 102")
print("Total: 358")

## Dataset Validation

The final men's dataset contains 358 completed international tournament matches:

- 256 FIFA World Cup matches from 2010, 2014, 2018, and 2022.
- 102 UEFA European Championship matches from Euro 2020 and Euro 2024.

This reproduces the historical match count used by the original prediction framework.

The 2026 FIFA World Cup is intentionally excluded from this dataset. It will be treated as an independent out-of-sample test set in the model evaluation stage.

Penalty shootouts are also preserved separately from regulation/extra-time goals. This distinction is important because the Poisson model predicts goals scored during the match, while tournament simulations must determine which team advances following a shootout.